In [2]:
import torch
import pandas as pd
from pathlib import  Path

In [3]:
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

In [4]:
device

'mps'

In [5]:
!nvidia-smi

zsh:1: command not found: nvidia-smi


In [6]:
from torch.utils.data import DataLoader,Dataset

In [7]:
# df = pd.read_csv("C:/Users/JAY/OneDrive/Desktop/Cheiron/Engine/data/train_hand_coordinates_dataset.csv")
csv_path = Path.cwd()/"data" / "train_hand_coordinates_dataset.csv"
df = pd.read_csv(csv_path)

In [8]:
df.head()

,label,x0,y0,z0,x1,y1,z1,x2,y2,z2,...,z17,x18,y18,z18,x19,y19,z19,x20,y20,z20
0,rock,0.489005,0.791847,1.228668e-07,0.658929,0.633890,0.103991,0.721733,0.506313,0.110695,...,-0.230397,0.338112,0.187038,-0.240333,0.390022,0.119882,-0.188500,0.428273,0.128910,-0.131621
1,rock,0.663417,0.757704,-4.471803e-07,0.528635,0.626601,0.138235,0.462252,0.537801,0.178455,...,-0.062645,0.809629,0.351067,0.003458,0.806058,0.335846,0.089098,0.798025,0.337741,0.159920
2,rock,0.501777,0.795207,-9.048249e-08,0.670459,0.627597,0.152369,0.733749,0.494408,0.173066,...,-0.261809,0.386238,0.184224,-0.245260,0.443748,0.125377,-0.173751,0.478180,0.134523,-0.108750
3,rock,0.523732,0.696118,-7.471434e-07,0.428049,0.605450,0.087049,0.409684,0.529910,0.125433,...,-0.021785,0.677395,0.381227,0.004800,0.669605,0.378249,0.056835,0.647914,0.403127,0.106704
4,rock,0.526907,0.683350,-6.492717e-07,0.420247,0.628914,0.067405,0.378990,0.569144,0.119739,...,0.079127,0.649809,0.393633,0.101652,0.640019,0.386109,0.138166,0.624287,0.404904,0.175719


In [9]:
from sklearn.preprocessing import LabelEncoder


class HandCoordinatesDataset(Dataset):
    def __init__(self, dataframe):
        # 1. Store the dataframe and reset index to ensure sequential access
        self.data = dataframe.copy().reset_index(drop=True)
        
        # 2. Extract features (all columns except 'label')
        self.features = self.data.drop(columns=['label']).values
        
        # 3. Extract and encode labels
        self.label_encoder = LabelEncoder()
        self.labels = self.label_encoder.fit_transform(self.data['label'].values)
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        # 4. Convert specific row to PyTorch tensors
        features = torch.tensor(self.features[idx], dtype=torch.float32)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        
        return features, label

In [10]:
from sklearn.model_selection import train_test_split


train_df, test_df = train_test_split(
    df, 
    test_size=0.2, 
    random_state=42, 
    stratify=df['label'] # Crucial for balanced classes
)

In [11]:
train_dataset = HandCoordinatesDataset(train_df)
test_dataset = HandCoordinatesDataset(test_df)

In [12]:
train_loader = DataLoader(train_dataset, 
                            batch_size=32, 
                            shuffle=True,
                            num_workers=0)

test_loader = DataLoader(test_dataset,
                        batch_size=32, 
                        shuffle=False,
                        num_workers=0)

In [13]:
from model_and_loops.model import RPSModel

model = RPSModel().to(device)

In [14]:
loss_fn = torch.nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

In [15]:
from model_and_loops.loops_for_the_model import training_loop

training_loop(train_loader =train_loader,  
                model=model,
                device=device,
                loss_fn=loss_fn, 
                optimizer=optimizer, 
                epochs=20, )

Epoch 1/20, Loss: 1.0705, Accuracy: 48.05%
Epoch 2/20, Loss: 0.8921, Accuracy: 89.04%
Epoch 3/20, Loss: 0.5172, Accuracy: 94.60%
Epoch 4/20, Loss: 0.2495, Accuracy: 96.62%
Epoch 5/20, Loss: 0.1212, Accuracy: 98.86%
Epoch 6/20, Loss: 0.0676, Accuracy: 99.22%
Epoch 7/20, Loss: 0.0430, Accuracy: 99.32%
Epoch 8/20, Loss: 0.0321, Accuracy: 99.53%
Epoch 9/20, Loss: 0.0244, Accuracy: 99.69%
Epoch 10/20, Loss: 0.0204, Accuracy: 99.64%
Epoch 11/20, Loss: 0.0199, Accuracy: 99.53%
Epoch 12/20, Loss: 0.0138, Accuracy: 99.79%
Epoch 13/20, Loss: 0.0124, Accuracy: 99.74%
Epoch 14/20, Loss: 0.0112, Accuracy: 99.84%
Epoch 15/20, Loss: 0.0097, Accuracy: 99.84%
Epoch 16/20, Loss: 0.0092, Accuracy: 99.79%
Epoch 17/20, Loss: 0.0081, Accuracy: 99.79%
Epoch 18/20, Loss: 0.0089, Accuracy: 99.84%
Epoch 19/20, Loss: 0.0074, Accuracy: 99.95%
Epoch 20/20, Loss: 0.0062, Accuracy: 99.90%


([1.070450692880349,
  0.89210202068579,
  0.5171888573247878,
  0.24946167930716373,
  0.12124535380328288,
  0.06763115665707432,
  0.043019534256614624,
  0.03213489310602184,
  0.024431337190211795,
  0.020356091596262497,
  0.019937128378231018,
  0.013831528365520424,
  0.012401604040174699,
  0.011232216838655657,
  0.009739566884446339,
  0.009249858586804667,
  0.008078438581013288,
  0.008854993544595286,
  0.007439712387391321,
  0.006188842993741092],
 [0.4805194805194805,
  0.8903896103896104,
  0.945974025974026,
  0.9662337662337662,
  0.9885714285714285,
  0.9922077922077922,
  0.9932467532467533,
  0.9953246753246753,
  0.9968831168831169,
  0.9963636363636363,
  0.9953246753246753,
  0.9979220779220779,
  0.9974025974025974,
  0.9984415584415585,
  0.9984415584415585,
  0.9979220779220779,
  0.9979220779220779,
  0.9984415584415585,
  0.9994805194805195,
  0.9989610389610389])